### 📚 Lesson 4.3: Leaky ReLU & ELU – Fixing Dead Neurons

---

**Mày đã học gì rồi?**

* Mày biết **Sigmoid** → cho output (0,1), hay dùng binary classification.
* Biết **Softmax** → multi-class, kết hợp với CrossEntropyLoss, chuẩn phân loại.
* Biết **ReLU** → đơn giản, nhanh, giảm vanishing gradient nhưng dễ dính **dead neurons** (output toàn 0).

---

#### ❓ Today's question

"ReLU rất ngon nhưng có nhược điểm là **dead neurons**. Làm sao để khắc phục chuyện này?"

---

#### 💡 Key ideas

* **Dead neurons** = khi input < 0, ReLU trả về 0, neuron ngưng học, có thể “chết” vĩnh viễn.
* Giải pháp:

  1. **Leaky ReLU**: thay vì output 0 khi x < 0, nó cho một slope nhỏ (ví dụ 0.01 \* x).

     $f(x) = \begin{cases} x & \text{if } x > 0 \\ \alpha x & \text{if } x \leq 0 \end{cases}$
     
     → neuron vẫn có chút gradient, không chết hẳn.

  2. **ELU (Exponential Linear Unit)**: smooth hơn, output âm không về 0 mà về giá trị gần -1.
     
     $f(x) = \begin{cases} x & \text{if } x > 0 \\ \alpha(e^x - 1) & \text{if } x \leq 0 \end{cases}$

     → giúp trung bình output gần 0, cải thiện learning.

👉 Nói ngắn: ReLU nhanh → Leaky ReLU chống chết → ELU học ổn định hơn.

---

#### 📝 Practice Problem

1. **Knowledge check:**

   * Giả sử dùng ReLU trong một mạng, nhiều neuron output = 0 mãi không đổi. Hiện tượng này gọi là gì?
   * Giải pháp nào trong activation function giúp xử lý vấn đề này?

2. **Coding (PyTorch):**

   * Huấn luyện một NN phân loại dataset 3 lớp (giống bài 4.2) nhưng thử thay:

     * `nn.ReLU()` → `nn.LeakyReLU(0.01)`
     * `nn.ReLU()` → `nn.ELU(alpha=1.0)`
   * So sánh loss sau training.

---

#### 🧩 Solution

**Question 1:**

- dead neurons
- Leaky ReLU and ELU, which allows negative outputs but smaller slope.

**Question 2:**

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from tqdm import tqdm
from sklearn.datasets import make_classification

In [2]:
# Data
X, Y = make_classification(
    n_samples=300,
    random_state=42,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_classes=3,
    n_clusters_per_class=1,
)
X = torch.tensor(X, dtype=torch.float32)
Y = torch.tensor(Y, dtype=torch.long)

In [3]:
# Training params
lr = 0.05
epochs = 5000

In [4]:
# Model: 2-layer NN
model_elu = nn.Sequential(nn.Linear(2, 16), nn.ELU(), nn.Linear(16, 3))
model_leaky = nn.Sequential(nn.Linear(2, 16), nn.LeakyReLU(), nn.Linear(16, 3))

In [5]:
# Loss + Optimizer
criterion = nn.CrossEntropyLoss()
optimizer_elu = optim.SGD(model_elu.parameters(), lr=lr)
optimizer_leaky = optim.SGD(model_leaky.parameters(), lr=lr)

In [6]:
for epoch in tqdm(range(epochs)):
    y_elu_pred = model_elu(X)
    loss_elu = criterion(y_elu_pred, Y)

    optimizer_elu.zero_grad()
    loss_elu.backward()
    optimizer_elu.step()

    y_leaky_pred = model_leaky(X)
    loss_leaky = criterion(y_leaky_pred, Y)

    optimizer_leaky.zero_grad()
    loss_leaky.backward()
    optimizer_leaky.step()

100%|██████████| 5000/5000 [00:02<00:00, 2412.09it/s]


In [7]:
print(f"Final ELU loss: {loss_elu.item():.6f}")
print(f"Final Leaky ReLU loss: {loss_leaky.item():.6f}")

Final ELU loss: 0.120078
Final Leaky ReLU loss: 0.104141
